# 05b | Probability: classic interview problems and simulation

Two goals:

1. Learn `scipy.stats`, the equivalent of R's `dnorm` / `pnorm` / `qnorm` / `rnorm` family;
2. Practice **verifying the answer to a probability problem by simulation**. If your derivation stalls in an interview, saying "I can write a quick simulation to check" and actually writing it earns real credit.

About 55 minutes.

## 1. scipy.stats: R's d / p / q / r

| R | scipy.stats | Meaning |
|---|---|---|
| `dnorm(x, mu, sd)` | `stats.norm.pdf(x, mu, sd)` | Density |
| `pnorm(x, mu, sd)` | `stats.norm.cdf(x, mu, sd)` | P(X ≤ x) |
| `1 - pnorm(x)` | `stats.norm.sf(x)` | P(X > x) |
| `qnorm(p)` | `stats.norm.ppf(p)` | Quantile |
| `rnorm(n)` | `stats.norm.rvs(size=n)` | Random draws |
| `dbinom(k, n, p)` | `stats.binom.pmf(k, n, p)` | Discrete distributions use `pmf` |
| `pt(x, df)` | `stats.t.cdf(x, df)` | |

The pattern: `stats.<distribution>.<method>(value, parameters...)`. Other distributions: `binom`, `poisson`, `expon`, `t`, `chi2`, `geom`, `uniform`.

⚠️ The exponential distribution is parameterized by **scale** (mean 1/λ): `stats.expon(scale=1/lam)`, the opposite of R's `rate`.

In [ ]:
import numpy as np
from scipy import stats

print(stats.norm.cdf(1.96))            # 0.975
print(stats.norm.ppf(0.975))           # 1.96
print(stats.binom.pmf(3, 10, 0.5))     # exactly 3 heads in 10 flips
print(stats.poisson.pmf(0, 3))         # e^-3
print(stats.t.ppf(0.975, df=9))        # t critical value

### ✏️ Exercise 1 (write it yourself)

Use `scipy.stats` to compute these three numbers:

- `p1`: the probability of **at least 8** heads in 10 flips of a fair coin;
- `p2`: $P(|Z| > 2)$ for a standard normal;
- `q`: the 97.5% quantile of a t distribution with 20 degrees of freedom.

In [ ]:
import numpy as np
from scipy import stats

# write your code below

<details>
<summary><b>Hint</b></summary>

`sf(k)` is P(X > k), so "at least 8" is `stats.binom.sf(7, 10, 0.5)`, or equivalently `1 - stats.binom.cdf(7, 10, 0.5)`. A two-sided probability is twice the one-sided one.

</details>

<details>
<summary><b>Answer</b></summary>

```python
import numpy as np
from scipy import stats

p1 = stats.binom.sf(7, 10, 0.5)        # P(X > 7) = P(X >= 8)
p2 = 2 * stats.norm.sf(2)
q = stats.t.ppf(0.975, df=20)
print(p1, p2, q)
```

</details>

## 2. Bayes as a function

Turn the disease-testing problem from 05a into code. Bayes problems keep coming back in interviews with different numbers, and a function lets you check your answer quickly.

### ✏️ Exercise 2 (fill in the blanks)

Complete the function: given prevalence `prior`, sensitivity `sens`, and specificity `spec`, return the probability of actually having the disease after a positive test.

In [ ]:
def posterior(prior, sens, spec):
    true_pos = sens * prior                  # sick and positive
    false_pos = (1 - ____) * (1 - prior)     # healthy but positive
    return true_pos / (true_pos + ____)

print(posterior(0.01, 0.99, 0.95))

<details>
<summary><b>Answer</b></summary>

```python
def posterior(prior, sens, spec):
    true_pos = sens * prior                  # sick and positive
    false_pos = (1 - spec) * (1 - prior)     # healthy but positive
    return true_pos / (true_pos + false_pos)

print(posterior(0.01, 0.99, 0.95))
```

</details>

## 3. Verifying by simulation: the Monty Hall problem

There are three doors, with a car behind one. You pick a door; the host (who knows where the car is) opens one of the other two doors that has no car behind it and asks whether you want to switch. Switching wins with probability 2/3, which many people don't believe. Let's simulate it:

The standard recipe for a simulation: **generate many random trials → check whether the event happened in each → take the mean**. Generate everything at once with NumPy where you can, instead of writing loops.

### ✏️ Exercise 3 (fill in the blanks)

Complete the simulation: `car` is the door with the car, `pick` is the door you chose first. If you stay, you win only if you picked the car; if you switch, you win only if your **first pick was wrong**.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n = 100_000
car = rng.integers(0, 3, n)       # which door has the car
pick = rng.integers(0, 3, n)      # which door you pick first

p_stay = (pick ____ car).mean()
p_switch = (pick ____ car).mean()
print(p_stay, p_switch)

<details>
<summary><b>Answer</b></summary>

```python
import numpy as np

rng = np.random.default_rng(0)
n = 100_000
car = rng.integers(0, 3, n)       # which door has the car
pick = rng.integers(0, 3, n)      # which door you pick first

p_stay = (pick == car).mean()
p_switch = (pick != car).mean()
print(p_stay, p_switch)
```

The code also gives you the intuition: your first pick is wrong with probability 2/3, and whenever it's wrong, switching always wins.

</details>

## 4. The birthday problem

The probability that at least two of n people share a birthday. First compute "everyone's birthday is different":

$$P(\text{all different}) = \frac{365}{365} \cdot \frac{364}{365} \cdots \frac{365 - n + 1}{365}$$

### ✏️ Exercise 4 (write it yourself)

Write a function `birthday(n)` that returns the probability that at least two of n people share a birthday (assume 365 days in a year). `birthday(23)` is about 0.507.

In [ ]:
def birthday(n):
    # write your code below
    pass

<details>
<summary><b>Hint</b></summary>

Running product: `p = 1.0`, `for i in range(n): p *= (365 - i) / 365`, then return `1 - p`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
def birthday(n):
    p_all_different = 1.0
    for i in range(n):
        p_all_different *= (365 - i) / 365
    return 1 - p_all_different
```

</details>

## 5. Expectation problems

**Method 1: the geometric distribution.** With success probability p, you expect to wait 1/p trials.

**Method 2: linearity of expectation.** Split the total into stages and add up each stage's expectation.

Example (the coupon collector problem): how many rolls of a die does it take, on average, to see all six faces? Once you've seen k faces, the chance of rolling a new one is (6−k)/6, so you expect 6/(6−k) rolls. Adding them all up:

$$\frac{6}{6} + \frac{6}{5} + \frac{6}{4} + \frac{6}{3} + \frac{6}{2} + \frac{6}{1} = 14.7$$

### ✏️ Exercise 5 (write it yourself)

Write a function `coupon_expected(n)`: there are n equally likely cards; return the expected number of draws needed to collect all n. `coupon_expected(6)` should be 14.7.

In [ ]:
def coupon_expected(n):
    # write your code below
    pass

<details>
<summary><b>Hint</b></summary>

Sum `n / (n - k)` over k = 0, 1, …, n−1.

</details>

<details>
<summary><b>Answer</b></summary>

```python
def coupon_expected(n):
    return sum(n / (n - k) for k in range(n))
```

</details>

**Method 3: state equations (or just simulate).** "How many coin flips do you expect until you get two heads in a row?"

Let $E_0$ be the expected number of flips starting from scratch, and $E_1$ the expected number still needed right after flipping one head:

$$E_0 = 1 + \tfrac12 E_1 + \tfrac12 E_0, \qquad E_1 = 1 + \tfrac12 \cdot 0 + \tfrac12 E_0$$

Solving gives $E_0 = 6$. Let's verify it by simulation. "Process-style" problems like this don't vectorize easily, so just write the loop.

### ✏️ Exercise 6 (write it yourself)

Write a function `flips_until_hh(rng)` that simulates **one** experiment: keep flipping a coin until two heads come up in a row, and return the total number of flips.

`rng.random() < 0.5` works as one coin flip (True means heads). The code below calls your function 20,000 times and averages the results; the answer should be close to 6.

In [ ]:
import numpy as np

def flips_until_hh(rng):
    # write your code below
    pass

rng = np.random.default_rng(0)
avg = np.mean([flips_until_hh(rng) for _ in range(20000)])
print(avg)

<details>
<summary><b>Hint</b></summary>

Two variables: `count` (how many flips so far) and `streak` (current run of heads). In `while streak < 2:`, do `count += 1` each time; on heads `streak += 1`, on tails `streak = 0`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
import numpy as np

def flips_until_hh(rng):
    count = 0
    streak = 0
    while streak < 2:
        count += 1
        if rng.random() < 0.5:
            streak += 1
        else:
            streak = 0
    return count

rng = np.random.default_rng(0)
avg = np.mean([flips_until_hh(rng) for _ in range(20000)])
print(avg)
```

</details>

## 6. See the CLT for yourself

The exponential distribution is right-skewed. Take samples of size n = 2 and n = 50, repeat each 10,000 times, and plot the distribution of the sample means.

(This app can display matplotlib plots. `plt.subplots(1, 2)` is like R's `par(mfrow = c(1, 2))`.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, n in zip(axes, [2, 50]):
    means = rng.exponential(1.0, size=(10000, n)).mean(axis=1)   # each row is one sample
    ax.hist(means, bins=50)
    ax.set_title(f"n = {n}")
plt.show()

**🗣 Say it out loud:** An interviewer asks you to "generate random numbers from some distribution, but all you have is Uniform(0, 1)". How do you do it?

<details>
<summary><b>Reference answer</b></summary>

Inverse transform sampling: if $U \sim \mathrm{Uniform}(0, 1)$ and $F$ is the target distribution's CDF, then $X = F^{-1}(U)$ follows the target distribution. For example, the exponential has $F(x) = 1 - e^{-\lambda x}$, so $X = -\ln(1 - U)/\lambda$. When the CDF is hard to invert, use rejection sampling. For a discrete distribution, cut [0, 1] into pieces sized by each value's probability and see which piece U lands in.

**Sample phrasing:** Inverse transform sampling: apply the inverse CDF to a uniform draw. For an exponential, that's minus log of one minus U over lambda. If the inverse CDF isn't available, use rejection sampling.

</details>

## Summary

- `stats.<distribution>.pdf / pmf / cdf / sf / ppf`;
- Simulation = generate many trials → check the event → take the mean;
- Three tools for expectation problems: the geometric distribution, splitting by linearity, state equations;
- When unsure of a derivation, verify it with a simulation.

On to **06a**.